# CrossLead Signal Analysis — Train vs Test Distributions

Loads the **recent properly-trained** RepNet CrossLead model:
- `cv_results/repnet_crosslead_2026-04-26_20-41-26/model_repnet_crosslead.pt`
- Optuna params: `lr=8.76e-4, dropout=0.0636`, 2-stage `(32, 64)`, kernels `(7, 5)`
- Trained on **unbalanced** dataset with **patient-grouped 80/20** split (no leakage)
- Reported test AUROC: 0.6965

Runs inference on the **training (dev) set** to assess whether the model learned to separate  
its own training data — the cleanest sanity check for whether there's any signal to learn at all.  
Then compares against the unseen test set.

**Decision rule:**
- Sharp bimodal on train + sharp on test → strong learnable signal  
- Sharp on train, mushy on test → overfitting, model memorized rather than generalized  
- Mushy on train and test → architecture/data has no real signal

In [28]:
import sys
from pathlib import Path
sys.path.insert(0, '../..')

import numpy as np
import pandas as pd
import torch
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots
from sklearn.metrics import roc_auc_score, average_precision_score, roc_curve
from torch.utils.data import DataLoader, TensorDataset

from src.models.repnet_crosslead import RepNetCrossLead
from src.data.dataset import load_seniordesign, split_holdout_grouped
from src.preprocessing.filters import BaselineWanderFilter, NotchFilter
from src.preprocessing.normalization import ZScoreNormalization

In [29]:
# Recent run: patient-grouped, unbalanced, Optuna best params
RUN_DIR    = Path('../../cv_results/repnet_crosslead_2026-04-26_20-41-26')
MODEL_PATH = RUN_DIR / 'model_repnet_crosslead.pt'
DATA_DIR   = '../../data/seniordesign_upload'
SEED       = 42

NET_PARAMS = dict(
    stage_filters = (32, 64),
    wide_kernel   = 7,
    narrow_kernel = 5,
    dropout       = 0.0636,
    n_heads       = 4,
)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')
print(f'Model:  {MODEL_PATH.resolve()}')

Device: cuda
Model:  C:\Users\email\PycharmProjects\repnet\cv_results\repnet_crosslead_2026-04-26_20-41-26\model_repnet_crosslead.pt


In [30]:
net = RepNetCrossLead(**NET_PARAMS).to(device)
net.load_state_dict(torch.load(MODEL_PATH, map_location=device))
net.eval()
n_params = sum(p.numel() for p in net.parameters())
print(f'Loaded. Parameters: {n_params:,}')

Loaded. Parameters: 116,802


In [31]:
# Load + same QC + same patient-grouped 80/20 split (seed=42)
X, y, patient_ids = load_seniordesign(DATA_DIR, return_patient_ids=True)

flat_mask = (X.std(axis=2) < 1e-4).any(axis=1)
try:
    nan_mask = np.isnan(patient_ids.astype(float))
except (ValueError, TypeError):
    nan_mask = np.array([str(p).strip() in ('', 'nan', 'None') for p in patient_ids])
keep = ~flat_mask & ~nan_mask
X, y, patient_ids = X[keep], y[keep], patient_ids[keep]

X, _ = BaselineWanderFilter(cutoff=0.5, order=4, fs=250.0).transform(X)
X, _ = NotchFilter(freq=60.0, Q=30.0, fs=250.0).transform(X)
X, _ = ZScoreNormalization(per_lead=True).transform(X)

X_dev, X_test, y_dev, y_test, g_dev, g_test = split_holdout_grouped(
    X, y, patient_ids, test_size=0.20, seed=SEED,
)

print(f'Dev (training) : N={len(y_dev)}   PE={int(y_dev.sum())}   Normal={int((y_dev==0).sum())}   ({100*y_dev.mean():.1f}% pos)')
print(f'Test (holdout) : N={len(y_test)}   PE={int(y_test.sum())}   Normal={int((y_test==0).sum())}   ({100*y_test.mean():.1f}% pos)')

Dev (training) : N=1747   PE=277   Normal=1470   (15.9% pos)
Test (holdout) : N=431   PE=58   Normal=373   (13.5% pos)


In [32]:
def infer(net, X, device, batch_size=64):
    Xt = torch.tensor(X, dtype=torch.float32)
    dl = DataLoader(TensorDataset(Xt), batch_size=batch_size, num_workers=0)
    out = []
    with torch.no_grad():
        for (xb,) in dl:
            logits = net(xb.to(device))
            out.append(torch.softmax(logits, dim=1)[:, 1].cpu().numpy())
    return np.concatenate(out)

probs_dev  = infer(net, X_dev,  device)
probs_test = infer(net, X_test, device)

auroc_dev  = roc_auc_score(y_dev,  probs_dev)
auroc_test = roc_auc_score(y_test, probs_test)
auprc_dev  = average_precision_score(y_dev,  probs_dev)
auprc_test = average_precision_score(y_test, probs_test)

print(f'Train (dev) — AUROC: {auroc_dev:.4f}    AUPRC: {auprc_dev:.4f}')
print(f'Test (held) — AUROC: {auroc_test:.4f}    AUPRC: {auprc_test:.4f}')
print(f'Generalization gap: {auroc_dev - auroc_test:+.4f}  (positive = train > test = some overfit)')

Train (dev) — AUROC: 0.8477    AUPRC: 0.5136
Test (held) — AUROC: 0.6965    AUPRC: 0.2787
Generalization gap: +0.1512  (positive = train > test = some overfit)


## Side-by-side distribution: train vs test

In [33]:
fig = make_subplots(rows=1, cols=2,
    subplot_titles=(
        f'Train set  (AUROC={auroc_dev:.3f}, N={len(y_dev)})',
        f'Test set  (AUROC={auroc_test:.3f}, N={len(y_test)})',
    ))

for col, (probs_, y_) in enumerate([(probs_dev, y_dev), (probs_test, y_test)], start=1):
    fig.add_trace(go.Histogram(
        x=probs_[y_ == 0], name='Normal', nbinsx=40,
        marker_color='steelblue', opacity=0.6,
        histnorm='probability density',
        showlegend=(col == 1), legendgroup='Normal',
    ), row=1, col=col)
    fig.add_trace(go.Histogram(
        x=probs_[y_ == 1], name='PE', nbinsx=40,
        marker_color='tomato', opacity=0.6,
        histnorm='probability density',
        showlegend=(col == 1), legendgroup='PE',
    ), row=1, col=col)
    fig.add_vline(x=0.5, line=dict(dash='dash', color='black'), row=1, col=col)

fig.update_layout(
    barmode='overlay',
    title='P(PE) distribution: training set vs holdout test',
    template='plotly_white', width=1100, height=460,
)
fig.update_xaxes(title_text='P(PE)')
fig.update_yaxes(title_text='Density', col=1)
fig.show()

## Summary statistics

In [34]:
def stats(probs_, y_):
    p_pe, p_norm = probs_[y_ == 1], probs_[y_ == 0]
    return {
        'PE_mean':         p_pe.mean(),
        'PE_median':       np.median(p_pe),
        'Normal_mean':     p_norm.mean(),
        'Normal_median':   np.median(p_norm),
        'mean_separation': p_pe.mean() - p_norm.mean(),
    }

summary = pd.DataFrame({
    'train': stats(probs_dev,  y_dev),
    'test':  stats(probs_test, y_test),
}).round(3)
summary

,train,test
PE_mean,0.745,0.605
PE_median,0.813,0.634
Normal_mean,0.376,0.410
Normal_median,0.339,0.377
mean_separation,0.369,0.194


## ROC curves overlaid

In [35]:
fpr_dv, tpr_dv, _ = roc_curve(y_dev,  probs_dev)
fpr_te, tpr_te, _ = roc_curve(y_test, probs_test)

fig3 = go.Figure()
fig3.add_trace(go.Scatter(x=fpr_dv, y=tpr_dv, mode='lines',
                          name=f'Train (AUC={auroc_dev:.3f})',
                          line=dict(color='steelblue', width=2)))
fig3.add_trace(go.Scatter(x=fpr_te, y=tpr_te, mode='lines',
                          name=f'Test  (AUC={auroc_test:.3f})',
                          line=dict(color='tomato', width=2)))
fig3.add_trace(go.Scatter(x=[0, 1], y=[0, 1], mode='lines', name='Random',
                          line=dict(dash='dash', color='gray')))
fig3.update_layout(title='ROC — Train vs Test',
                   xaxis_title='FPR', yaxis_title='TPR',
                   template='plotly_white', width=600, height=520)
fig3.show()

## Patient-level aggregation (test set)

Mean P(PE) across recordings per patient. If averaging boosts AUROC noticeably, recording-level noise is the bottleneck.

In [36]:
uniq = np.unique(g_test)
pat_probs, pat_labels = [], []
for pid in uniq:
    mask = g_test == pid
    pat_probs.append(probs_test[mask].mean())
    pat_labels.append(int(y_test[mask][0]))
pat_probs  = np.array(pat_probs)
pat_labels = np.array(pat_labels)

pat_auroc = roc_auc_score(pat_labels, pat_probs)
print(f'Patient-level test AUROC: {pat_auroc:.4f}  (N={len(uniq)} patients,  '
      f'PE={int(pat_labels.sum())}, Normal={int((pat_labels==0).sum())})')
print(f'Sample-level test AUROC : {auroc_test:.4f}  → Δ={pat_auroc - auroc_test:+.4f}')

fig4 = go.Figure()
fig4.add_trace(go.Histogram(x=pat_probs[pat_labels == 0], name='Normal', nbinsx=30,
                            marker_color='steelblue', opacity=0.6,
                            histnorm='probability density'))
fig4.add_trace(go.Histogram(x=pat_probs[pat_labels == 1], name='PE', nbinsx=30,
                            marker_color='tomato', opacity=0.6,
                            histnorm='probability density'))
fig4.update_layout(barmode='overlay',
                   title=f'Patient-level mean P(PE) on test (AUROC={pat_auroc:.3f})',
                   xaxis_title='Mean P(PE) per patient', yaxis_title='Density',
                   template='plotly_white', width=820, height=460)
fig4.add_vline(x=0.5, line=dict(dash='dash', color='black'))
fig4.show()

Patient-level test AUROC: 0.7512  (N=277 patients,  PE=35, Normal=242)
Sample-level test AUROC : 0.6965  → Δ=+0.0548
